# Search candidate decks with a frozen policy

This notebook proposes a small set of legal changes, evaluates them against your opponent deck, and checks the selected deck on separate game seeds. It searches the implemented candidate pool. Results do not establish the best Standard deck.

Use episode receipts from notebook 11 for both model inputs. Set the budgets below before running. Search and held-out seeds must differ. Capped games prevent a completed selection result.


In [ ]:
from pathlib import Path
import json, subprocess, sys, uuid
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/search_candidate_decks.py').exists()), None)
if ROOT is None: raise RuntimeError('Open from the death-knight-lab folder.')
CANDIDATE_RECEIPT = None
OPPONENT_RECEIPT = None
ROUNDS = 1
NEIGHBORS = 2
SEARCH_SEEDS = [920001, 920002]
HELDOUT_SEEDS = [930001, 930002]
NEIGHBOR_SEED = 940001
MAX_ACTIONS = 500
MAX_GAMES = 20
if CANDIDATE_RECEIPT is None or OPPONENT_RECEIPT is None:
    raise ValueError('Set both episode receipt paths from notebook 11.')
left = json.loads(Path(CANDIDATE_RECEIPT).read_text())
right = json.loads(Path(OPPONENT_RECEIPT).read_text())
plan = dict(schema='candidate-deck-search-v1', initial_deck=left['replay_only_decks'][0],
            opponents={'opponent': right['replay_only_decks'][1]},
            checkpoints=[left['checkpoint'], right['checkpoint']],
            rounds=ROUNDS, neighbors=NEIGHBORS, search_seeds=SEARCH_SEEDS,
            heldout_seeds=HELDOUT_SEEDS, neighbor_seed=NEIGHBOR_SEED,
            max_actions=MAX_ACTIONS, max_games=MAX_GAMES)
folder = ROOT/'runs/search_plans'
folder.mkdir(parents=True, exist_ok=True)
path = folder/(uuid.uuid4().hex+'.json')
path.write_text(json.dumps(plan, indent=2)+'\n')
command = [sys.executable, str(ROOT/'tools/search_candidate_decks.py'), '--engine-root',
           str(ROOT/'staging/rebased-88'), '--plan', str(path), '--output',
           str(ROOT/'runs/deck_search'), '--experimental']
completed = False
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            try: item = json.loads(line)
            except ValueError:
                print(line.rstrip()); continue
            if item['event']=='started': print('Reserved game budget:', item['reserved_games'])
            elif item['event']=='round': print('Round', item['round'], 'finished; games:', item['games'])
            elif item['event']=='complete':
                completed = True
                print('Saved search:', item['manifest'])
                print('Finished evaluation:', item['completed'])
        code = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if code or not completed: raise RuntimeError('Search did not finish; inspect the output above.')
